In [58]:
from typing import Annotated
from langchain_groq import ChatGroq
from langgraph.graph import START,StateGraph,END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict
from langchain_core.messages import BaseMessage,AIMessage,SystemMessage,HumanMessage

In [59]:
load_dotenv()

True

In [60]:
class chatstate(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [61]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=1000,
)

In [62]:

def chat_node(state:chatstate):
    decision=interrupt({
        "type":"approval",
        "reason":"model is about to answer a user question",
        "question":state["messages"][-1].content,
        "instruction":"approve this question ? yes/no"

    })
    if decision["approved"]=="no":
        return {"messages":[AIMessage(content="not approved")]}
    else:
        response=llm.invoke(state["messages"])  
        return {"messages": [response]}  


In [63]:
builder=StateGraph(chatstate)

builder.add_node("chat",chat_node)

builder.add_edge(START,"chat")
builder.add_edge("chat",END)

checkpoint=MemorySaver()

app=builder.compile(checkpointer=checkpoint)



In [64]:
config={"configurable":{"thread_id":"1234"}}


result=app.invoke({
    "messages":[
        ("user", "explain gradent descent for interview")
    ]
},config=config)


In [65]:
# 1. Extract the interrupt payload dictionary
interrupt_data = result["__interrupt__"][0].value

# 2. Display the question and prompt for approval
print(f"Backend reason : {interrupt_data.get('reason')}")
print(f"User question  : {interrupt_data.get('question')}")

user_input = input(f"{interrupt_data.get('instruction')} (yes/no): ").strip().lower()

# 3. Resume the graph with the decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

# 4. View the final response
print("\n--- Model Response ---")
print(final_result["messages"][-1].content)


Backend reason : model is about to answer a user question
User question  : explain gradent descent for interview

--- Model Response ---
not approved


{'messages': [HumanMessage(content='explain gradent descent in simple terms for interview', additional_kwargs={}, response_metadata={}, id='3b27d3e8-4f44-4bcc-93c3-ffd5e797f3c5'), AIMessage(content='**Gradient Descent – A Simple Interview‑Ready Explanation**\n\n---\n\n### 1. The Core Idea (in one sentence)\nGradient descent is an iterative method that **tunes the parameters of a model by repeatedly moving them a little bit in the direction that most reduces the error**.\n\n---\n\n### 2. Everyday Analogy  \nImagine you’re standing on a foggy hill at night, and you want to reach the lowest point (the valley).  \n- **You can’t see the whole landscape**, only the slope right under your feet.  \n- You feel the slope (the *gradient*) and take a small step downhill.  \n- You repeat: feel the new slope, step again, until the ground feels flat (the gradient is zero).  \n\nThat’s exactly what gradient descent does, but in a high‑dimensional “parameter space” instead of a physical hill.\n\n---\n\